In [ ]:
#| default_exp datasources.srd
%load_ext autoreload
%autoreload 2

import sys,os
from pathlib import Path

In [ ]:
# Insert in Path Project Directory
sys.path.insert(0, str(Path().cwd().parent))
os.chdir(Path.cwd().parent / 'extracao')

# Radiodifusão
> Módulo para encapsular a extração e processamento das estações de Radiodifusão Pública

In [ ]:
#| export
import os
from decimal import Decimal

import pandas as pd
from dotenv import find_dotenv, load_dotenv

from extracao.constants import (
    BW_MAP,
    COLS_SRD,
    DICT_SRD,
    MONGO_SRD,
    PROJECTION_SRD,
    RELATORIO_SRD,
)

from extracao.datasources.mosaico import Mosaico

In [ ]:
#| export
load_dotenv(find_dotenv(), override=True)
pd.options.mode.copy_on_write = True

True

In [ ]:
#| hide: true
#| eval:false
__file__ = Path.cwd().parent / 'extracao' / 'datasources.py'

In [ ]:
#| export
MONGO_URI = os.environ.get("MONGO_URI")

In [ ]:
#| export
class SRD(Mosaico):
    """Class to encapsulate the Radio Broadcasting Service extraction logic"""

    def __init__(
        self, mongo_uri: str = MONGO_URI, limit: int = 0, read_cache: bool = False
    ) -> None:
        super().__init__(mongo_uri, read_cache)
        self.limit = limit

    @property
    def stem(self):
        return "srd"

    @property
    def collection(self):
        return "srd"

    @property
    def query(self):
        return MONGO_SRD

    @property
    def projection(self):
        return PROJECTION_SRD

    @property
    def columns(self):
        return COLS_SRD

    @property
    def cols_mapping(self):
        return DICT_SRD

    def extraction(self) -> pd.DataFrame:
        """Extracts the data from the MongoDB database and returns a DataFrame"""
        pipeline = [{"$match": self.query}, {"$project": self.projection}]
        if self.limit > 0:
            pipeline.append({"$limit": self.limit})
        return self._extract(self.collection, pipeline)

    def _extract(self, collection: str, pipeline: list):
        if self.read_cache:
            return self._read(f"{self.stem}_raw")
        client = self.connect()
        database = client[self.database]
        db_collection = database[collection]
        df = pd.DataFrame(
            list(db_collection.aggregate(pipeline)), copy=False,
        )
        for row in df.itertuples():
            if isinstance(row.loctx, (list, tuple)) and len(row.loctx) == 2:
                df.loc[row.Index, ["LongTX", "LatTX"]] = str(row.loctx[0]), str(row.loctx[1])
            if isinstance(row.locpb, (list, tuple)) and len(row.locpb) == 2:
                df.loc[row.Index, ["LongPB", "LatPB"]] = str(row.locpb[0]), str(row.locpb[1])
        c = df.LatPB.isna() & df.LatTX.notna()
        df.loc[c, ["LongPB", "LatPB"]] = df.loc[c, ["LongTX", "LatTX"]] 
        
        # Substitui strings vazias, espaços e listas vazias por nulo
        df = df.astype("string", copy=False).replace(r"^\s*$|^\[\]$", pd.NA, regex=True)
        # Create the Log Column
        df["Log"] = "[]"
        return df
    

    def _format(
        self,
        df: pd.DataFrame,  # DataFrame com o resultantes do banco de dados
    ) -> pd.DataFrame:  # DataFrame formatado
        """Formats, cleans and standardizes the queried data from the database"""
        df = df.rename(columns=self.cols_mapping)
        status = df.Status.str.contains("-C1$|-C2$|-C3$|-C4$|-C7|-C98$", na=False)

        # Discard inactive statuses
        # discarded = df[~status].copy()
        # processing = 'Status não considerado para fins de monitoração'
        # Mosaico.register_log(discarded, processing, 'Status')
        # self.append2discarded(discarded)

        df[status].reset_index(drop=True, inplace=True)

        # Discard null frequencies
        discarded = df[df.Frequência.isna()].copy()
        processing = "Valor Nulo."
        Mosaico.register_log(discarded, processing, "Frequência")

        df.dropna(subset="Frequência", ignore_index=True, inplace=True)  # type: ignore
        df["Frequência"] = (
            df.Frequência.astype("string", copy=False)
            .str.replace(",", ".")
            .astype("float")
        )

        df.loc[df["Serviço"] == "205", "Frequência"] = df.loc[
            df["Serviço"] == "205", "Frequência"
        ].apply(lambda x: float(Decimal(x) / Decimal(1000)))
        df["Validade_RF"] = df.Validade_RF.astype("string", copy=False).str.slice(0, 10)

        df["Fonte"] = "MOSAICO-SRD"
        df["Fonte"] = df["Fonte"].astype("string", copy=False)

        df["Designação_Emissão"] = df.Serviço.fillna("").map(BW_MAP)
        df = Mosaico.split_designacao(df)

        df["Multiplicidade"] = "1"
        df["Multiplicidade"] = df["Multiplicidade"].astype("string", copy=False)

        df["Padrão_Antena(dBd)"] = df["Padrão_Antena(dBd)"].str.replace("None", "0")
        df["Potência_Transmissor(W)"] = pd.to_numeric(
            df["Potência_Transmissor(W)"], errors="coerce"
        ).astype("float")
        df["Potência_Transmissor(W)"] = (
            df["Potência_Transmissor(W)"]
            .apply(lambda x: float(Decimal(1000) * Decimal(x)))
            .astype("string", copy=False)
        )

        df["Relatório_Canal"] = df.apply(
            lambda row: RELATORIO_SRD.format(row.loc["Id"], row.loc["Status"]), axis=1
        ).astype("string", copy=False)

        return df.loc[:, self.columns]
